Import and Load

In [1]:
import pandas as pd
import numpy as np
import joblib

best_model = joblib.load('../models/best_model.pkl')
print("Model loaded")

Model loaded


Predict All Car Parks for a Given Time

In [2]:
def predict_all_parks(hour, day_encoded, num_parks=18, prev_occupancy=50.0):
    results = []
    for park_id in range(num_parks):
        sample = pd.DataFrame([{
            'Hour': hour,
            'DayOfWeek_encoded': day_encoded,
            'ParkID_encoded': park_id,
            'OccupancyRate_lag1': prev_occupancy
        }])
        pred = best_model.predict(sample)[0]
        results.append({'ParkID': park_id, 'PredictedOccupancy': round(pred, 1)})

    return pd.DataFrame(results).sort_values('PredictedOccupancy')

# Test: Wednesday 8AM
predictions = predict_all_parks(hour=8, day_encoded=2)
print(predictions)

    ParkID  PredictedOccupancy
5        5           20.600000
6        6           22.100000
7        7           22.100000
4        4           25.900000
0        0           30.400000
11      11           34.099998
1        1           35.799999
2        2           43.099998
17      17           44.000000
3        3           44.200001
9        9           44.900002
8        8           45.299999
15      15           50.500000
10      10           52.200001
14      14           53.099998
13      13           54.700001
12      12           55.799999
16      16           56.000000


Recommendation Function

In [3]:
def recommend_parking(hour, day_encoded):
    predictions = predict_all_parks(hour, day_encoded)
    best = predictions.iloc[0]
    worst = predictions.iloc[-1]

    # CO2 estimate: every 1% extra occupancy = ~1.2g extra CO2 from searching
    co2_saved = round((worst['PredictedOccupancy'] - best['PredictedOccupancy']) * 1.2, 1)

    print("=" * 40)
    print(f"Recommended Car Park : Park {int(best['ParkID'])}")
    print(f"Predicted Occupancy  : {best['PredictedOccupancy']}%")
    print(f"Estimated CO2 Saved  : {co2_saved}g vs busiest park")
    print("=" * 40)
    print("\nAll Parks Ranked:")
    print(predictions.to_string(index=False))

    return best, co2_saved

# Test it
recommend_parking(hour=8, day_encoded=2)

Recommended Car Park : Park 5
Predicted Occupancy  : 20.600000381469727%
Estimated CO2 Saved  : 42.5g vs busiest park

All Parks Ranked:
 ParkID  PredictedOccupancy
      5           20.600000
      6           22.100000
      7           22.100000
      4           25.900000
      0           30.400000
     11           34.099998
      1           35.799999
      2           43.099998
     17           44.000000
      3           44.200001
      9           44.900002
      8           45.299999
     15           50.500000
     10           52.200001
     14           53.099998
     13           54.700001
     12           55.799999
     16           56.000000


(ParkID                 5.0
 PredictedOccupancy    20.6
 Name: 5, dtype: float64,
 42.5)

Save Recommendation Function for App

In [4]:
import joblib

# Bundle model and function info together
joblib.dump(best_model, '../models/best_model.pkl')
print("Ready for Streamlit app")

Ready for Streamlit app
